# D240 — Introduction to Apache Hive

**Environment used in this lesson:** WSL/Ubuntu, Hadoop 3.3.6, Hive 4.0.1, Java 11, MySQL metastore, and MapReduce on YARN.

> Run the shell commands in an Ubuntu/WSL terminal. The notebook code cells use `%%bash`, which works only when the notebook kernel can access the same WSL/Linux environment. Commands that start services are intentionally separated so you can observe and troubleshoot each component.

For Further reading, refer to https://hive.apache.org/development/desingdocs/design/

## 1. What is Hive?

Apache Hive is a **distributed data-warehouse system built on Hadoop**. It lets analysts describe data with tables and query it using HiveQL, a SQL-like language. Hive translates HiveQL into work that an execution engine runs against data stored in HDFS.

Hive is designed for batch analytics, summarization, ETL, and large scans—not low-latency row-by-row transactions. It uses **schema-on-read**: the data can already exist in HDFS, and Hive applies table structure when a query reads it.

| Layer | Responsibility |
|---|---|
| Hive | SQL interface, schema, query planning, optimization |
| HDFS | Distributed storage for table data |
| YARN | Cluster resources and job scheduling |
| MapReduce | Execution engine selected in this training setup |
| MySQL | Durable storage for Hive metadata—not the table rows |

## 2. Architecture

![Apache Hive architecture](hive_architecture.png)

A query enters through Beeline/JDBC, reaches HiveServer2, and is parsed, compiled, and optimized by the driver. The metastore supplies table and partition metadata. The executor submits work to MapReduce/YARN, which reads or writes the actual data in HDFS.

### Main components

- **Beeline:** command-line JDBC client; the preferred interactive client for HiveServer2.
- **HiveServer2 (HS2):** accepts authenticated client sessions and executes HiveQL requests. Default binary-transport port: `10000`; web UI commonly uses `10002`.
- **Driver:** manages the query lifecycle. The parser checks syntax, the compiler builds a logical plan, the optimizer improves it, and the executor coordinates execution.
- **Metastore service:** exposes metadata such as databases, tables, columns, locations, and partitions. Default Thrift port: `9083`.
- **Metastore database (MySQL here):** persists metadata. Actual table data remains in HDFS.
- **HDFS:** stores managed-table warehouse files and external-table files.
- **YARN + MapReduce:** allocates resources and runs the physical jobs in this lab.

## 3. Prerequisite spot-check

This notebook assumes your detailed Hadoop and Hive setup has already been completed. These checks confirm that the expected binaries and environment variables are available.

In [ ]:
%%bash
java -version
hadoop version | head -5
hive --version
printf 'HADOOP_HOME=%s\nHIVE_HOME=%s\n' "$HADOOP_HOME" "$HIVE_HOME"

Expected major versions are Java 11, Hadoop 3.3.6, and Hive 4.0.1. If a command is not found, open a new WSL shell or run `source ~/.bashrc`, then check the relevant installation path and `PATH`.

## 4. Start and observe the Hadoop layer

Start HDFS, YARN, and the MapReduce JobHistory Server in the WSL terminal. Re-running a start command when a daemon is already running may print an informational message.

In [ ]:
%%bash
start-dfs.sh
start-yarn.sh
mapred --daemon start historyserver
jps

`jps` should show `NameNode`, `DataNode`, `SecondaryNameNode`, `ResourceManager`, `NodeManager`, and `JobHistoryServer` (plus `Jps`). Confirm health rather than relying only on process names.

In [ ]:
%%bash
hdfs dfsadmin -report | grep -E 'Live datanodes|Name:|Decommission Status'
yarn node -list
ss -lnt | grep -E ':(9000|9870|8088|8042|19888|10020)\b' || true

Expected: one live DataNode and one running YARN NodeManager. Useful web UIs are NameNode `http://localhost:9870`, ResourceManager `http://localhost:8088`, and JobHistory `http://localhost:19888`. Port `9000` is Hadoop RPC, not a browser UI.

## 5. Verify Hive storage locations

Hive needs writable temporary and warehouse locations. These commands are idempotent for a single-node training environment.

In [ ]:
%%bash
hdfs dfs -mkdir -p /tmp /tmp/hive /user/hive/warehouse /user/hive/external
hdfs dfs -chmod 1777 /tmp /tmp/hive
hdfs dfs -chown "$USER:$(id -gn)" /user/hive/warehouse /user/hive/external
hdfs dfs -chmod 775 /user/hive/warehouse /user/hive/external
hdfs dfs -ls -d /tmp /tmp/hive /user/hive/warehouse /user/hive/external

## 6. Check the metastore schema and start Hive services

The schema should already have been initialized once with `schematool -dbType mysql -initSchema --verbose`. Do **not** repeatedly initialize it. Use `-info` for a safe status check. Then start the metastore before HiveServer2.

In [ ]:
%%bash
schematool -dbType mysql -info
mkdir -p "$HOME/hive-logs"
nohup hive --service metastore > "$HOME/hive-logs/metastore.log" 2>&1 &
echo 'Metastore start requested; check port 9083 after it initializes.'

In [ ]:
%%bash
ss -lnt | grep ':9083' || { echo 'Metastore is not listening yet; recent log:'; tail -80 "$HOME/hive-logs/metastore.log"; }

In [ ]:
%%bash
nohup hiveserver2 > "$HOME/hive-logs/hiveserver2.log" 2>&1 &
echo 'HiveServer2 start requested; check ports 10000 and 10002 after it initializes.'

In [ ]:
%%bash
ss -lnt | grep -E ':(9083|10000|10002)\b' || true
jps | grep -E 'HiveMetaStore|HiveServer2|RunJar' || true
echo 'If port 10000 is absent, inspect:'
echo "  tail -100 $HOME/hive-logs/hiveserver2.log"

## 7. Connect with Beeline

For an interactive session, run this in a WSL terminal:

```bash
beeline -u 'jdbc:hive2://localhost:10000/default' -n "$USER"
```

Inside Beeline, SQL statements end with a semicolon. The next cells show scripts that can also be sent non-interactively with Beeline.

## 8. First database, table, insert, and query

A database is a namespace. A table maps columns and data types to files. This first table is **managed**, so Hive controls its default warehouse location and lifecycle.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/default' -n "$USER" --silent=true -e "
SHOW DATABASES;
CREATE DATABASE IF NOT EXISTS training;
USE training;
CREATE TABLE IF NOT EXISTS test_message (id INT, message STRING);
INSERT INTO test_message VALUES (1, 'Hive is working');
SELECT * FROM test_message;
DESCRIBE FORMATTED test_message;
"

Expected query row: `1    Hive is working`. In `DESCRIBE FORMATTED`, locate `Location` to see the table's HDFS warehouse path. Re-running the insert adds another row; use `TRUNCATE TABLE training.test_message;` first when you need repeatable output.

## 9. Aggregation and the execution engine

Disabling fetch-task conversion forces this small query through the configured MapReduce engine, making the Hadoop execution path visible for learning. It is not a performance recommendation.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/training' -n "$USER" --silent=true -e "
SET hive.execution.engine;
SET hive.execution.engine=mr;
SET hive.fetch.task.conversion=none;
DROP TABLE IF EXISTS sales;
CREATE TABLE sales (city STRING, amount INT);
INSERT INTO sales VALUES ('Bengaluru',100),('Hyderabad',200),('Bengaluru',300),('Chennai',150);
SELECT city, SUM(amount) AS total FROM sales GROUP BY city ORDER BY city;
"

Expected totals: Bengaluru 400, Chennai 150, Hyderabad 200. While it runs, another terminal can show submitted jobs with `yarn application -list -appStates ALL`.

## 10. Managed vs external tables

- A **managed table** lets Hive manage both metadata and the table's warehouse files. Dropping it normally removes both.
- An **external table** points to data whose lifecycle is managed outside Hive. Dropping it removes Hive metadata but normally preserves files at `LOCATION`.

External tables are useful when files are shared with other Hadoop tools or must survive table-definition changes.

In [ ]:
%%bash
printf 'order_id,city,amount\n1,Bengaluru,100\n2,Chennai,150\n' > /tmp/orders.csv
hdfs dfs -mkdir -p /user/hive/external/orders
hdfs dfs -put -f /tmp/orders.csv /user/hive/external/orders/
beeline -u 'jdbc:hive2://localhost:10000/training' -n "$USER" --silent=true -e "
CREATE EXTERNAL TABLE IF NOT EXISTS external_orders (order_id INT, city STRING, amount INT)
ROW FORMAT DELIMITED FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION '/user/hive/external/orders'
TBLPROPERTIES ('skip.header.line.count'='1');
SELECT * FROM external_orders;
"

## 11. Partitions and `EXPLAIN`

A partition divides a table into directory-like slices, often by date or region. A filter on the partition column lets Hive skip unrelated files (**partition pruning**). Avoid creating extremely high-cardinality partitions.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/training' -n "$USER" --silent=true -e "
DROP TABLE IF EXISTS daily_sales;
CREATE TABLE daily_sales (city STRING, amount INT) PARTITIONED BY (sale_date STRING);
INSERT INTO daily_sales PARTITION (sale_date='2026-08-18') VALUES ('Bengaluru',100),('Chennai',150);
SHOW PARTITIONS daily_sales;
EXPLAIN SELECT city, SUM(amount) FROM daily_sales
WHERE sale_date='2026-08-18' GROUP BY city;
"

Read `EXPLAIN` from the scan outward: table/partition scan → operators such as filter and group-by → stages/tasks. Confirm that the selected partition is restricted to `sale_date=2026-08-18`. Use `EXPLAIN` before running expensive queries.

## 12. Troubleshooting checklist

1. Run `jps`; verify Hadoop daemons and Hive Java processes.
2. Run `hdfs dfsadmin -report`; verify `Live datanodes (1)`.
3. Run `yarn node -list`; verify one running node.
4. Check listeners: `ss -lnt | grep -E ':(9083|10000|10002)\b'`.
5. Read `~/hive-logs/metastore.log` and `~/hive-logs/hiveserver2.log`.
6. Check the newest Hive log under `/tmp/$USER/` or `$HOME`.
7. Verify MySQL is running and `schematool -dbType mysql -info` succeeds.
8. Confirm HDFS directory ownership and permissions.

Do not use `pkill -f` as the routine shutdown method; it can match unintended Java processes.

In [ ]:
%%bash
echo '--- Java processes ---'
jps
echo '--- Hive ports ---'
ss -lnt | grep -E ':(9083|10000|10002)\b' || true
echo '--- Recent HiveServer2 log ---'
tail -40 "$HOME/hive-logs/hiveserver2.log" 2>/dev/null || true

## 13. Safe shutdown

Exit Beeline with `!quit`. If Hive services were started by these notebook commands, capture their PIDs when starting them in future automation. For this manual lab, identify the exact Hive PIDs with `jps -lv` and terminate only those PIDs. Then stop the Hadoop services in dependency order:

```bash
mapred --daemon stop historyserver
stop-yarn.sh
stop-dfs.sh
jps
```

You now have the complete path: **HiveQL client → Hive services and metadata → query plan → YARN/MapReduce → HDFS data**.